# Validation harness for the causal-SDP scaffold

This notebook is an executable harness for the minimal SDP scaffold. It validates the cone wiring on a known causally separable white-noise process. The ideal quantum-switch benchmark is intentionally not claimed complete yet.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from deltawkrel.projectors import ProcessDims, white_noise_process, d_output
from deltawkrel.sdp import solve_cs_robustness_scaffold
from deltawkrel.switch_models import ideal_quantum_switch

dims = ProcessDims(2,2,2,2)
W_white = white_noise_process(dims)
print("Tr(W_white)=", W_white.trace(), "d_O=", d_output(dims))

## CS sanity benchmark: white noise
White noise is causally separable, so the scaffold robustness should be zero within solver tolerance.

In [ ]:
diag = solve_cs_robustness_scaffold(W_white, dims=dims, solver="CLARABEL")
print(diag)
assert diag.status in {"optimal", "optimal_inaccurate"}
assert diag.objective_value < 1e-5

## Required next step: ideal quantum switch
The ideal-switch process is not faked. The function raises `NotImplementedError` until the convention is implemented from the target reference and validated against a known published benchmark.

In [ ]:
try:
    ideal_quantum_switch(dims)
except NotImplementedError as exc:
    print("Expected pending benchmark:", exc)

## Export diagnostic

In [ ]:
out = ROOT / "exports"
out.mkdir(exist_ok=True)
with open(out / "sdp_scaffold_white_noise_diagnostics.json", "w", encoding="utf-8") as f:
    json.dump(diag.to_dict(), f, indent=2)
print(out / "sdp_scaffold_white_noise_diagnostics.json")